In [ ]:
%%writefile .envOPENROUTER_API_KEY="${OPENROUTER_API_KEY}"

In [2]:
%pip install beautifulsoup4


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from dotenv import load_dotenv
import os

In [4]:
load_dotenv()

True

In [5]:
OPENROUTER_API_KEY= os.getenv("OPENROUTER_API_KEY")

In [6]:
OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"
MODEL="openrouter/free"

openrouter=OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL, timeout=120, max_retries=5)

In [7]:
messages=[{'role':'user', 'content':"Hi there"}]
response=openrouter.chat.completions.create(model=MODEL, messages=messages)



In [8]:
print(response.model)
print(response.choices[0].message.content)

nvidia/nemotron-3-ultra-550b-a55b:free
Hi there! How can I help you today?


In [9]:
#harness the tool
MAZE_URL="https://maze.edwarddonner.com"

def read_web_page(url):
  """ Read a web page in the maze and return the text on the page without the HTML"""
  print(f"Reading the web page: {url}")
  if not url.startswith(MAZE_URL):
    return f"Sorry this tool can only read pages int he maze, Please ask for a URL that starts with {MAZE_URL}"
  response=requests.get(url, timeout=20)
  page=BeautifulSoup(response.text, "html.parser")
  return page.body.get_text(separator="\n", strip=True)

In [10]:
print(read_web_page("https://maze.edwarddonner.com"))

Reading the web page: https://maze.edwarddonner.com
The Maze
This page is for an AI that can read web pages. Your goal is to find the way out of a small maze and bring back the message that is waiting there. Every location in the maze is a web page that describes the room and lists its exits.
Step 1 is the entrance:
Location 1
https://maze.edwarddonner.com/1
Start with the maze now. Open the entrance, then follow the exits one page at a time. Some passages are dead ends, so go back and try another way when you reach one.
Keep the person you are working with updated as you go. After each location, say where you are and what you found there, and show a mini-map of the maze as you have seen it so far, drawn in text with the rooms you have visited and the exits you know about. When you reach the way out, give the message and a final map of your whole journey.
If you are a person reading this, give this address to ChatGPT or Claude and ask it to do what the page says.


In [11]:
read_web_page_tool = {
    "type": "function",
    "function": {
        "name": "read_web_page",
        "description": "Read a web page in the maze and return the text on the page without the HTML. Only works for URLs that start with https://maze.edwarddonner.com",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {
                    "type": "string",
                    "description": "The full URL of the maze page to read"
                }
            },
            "required": ["url"]
        }
    }
}

tools=[read_web_page_tool]


In [12]:
def handle_tool_call(tool_call):
  """ Run the tool the model asked for and package the result as a message"""
  try:
    arguments=json.loads(tool_call.function.arguments)
    result=read_web_page(**arguments)
  except Exception as error:
    result=f"The tool call failed: {error}"
  return {"role":"tool", "tool_call_id":tool_call.id, "content":result}


In [13]:
type(handle_tool_call)

function

In [14]:
messages=[{"role":"user", "content":"Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now without going into too much detal "}]
response=openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message=response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason:{response.choices[0].finish_reason}")
print(message.tool_calls)

Answered by nvidia/nemotron-3.5-super-vl:free
Finish reason:tool_calls
[ChatCompletionMessageFunctionToolCall(id='call-365ec444-0afe-4dfa-8a27-8dfb772ff4fd', function=Function(arguments='{"url":"https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)]


In [15]:
messages.append(message)
for tool_call in message.tool_calls:
  messages.append(handle_tool_call(tool_call))
response=openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print(f"Answered by {response.model}")
print(response.choices[0].message.content)

Reading the web page: https://maze.edwarddonner.com
Answered by nvidia/nemotron-3.5-lightning:free
The page you asked about is the “Maze” intro page hosted at **https://maze.edwarddonner.com**.  
Here’s what it says, in a nutshell:

- **Purpose:** It’s a small maze‑exploration challenge designed for an AI (or anyone) to read a series of web pages, each describing a room and listing the possible exits.
- **Goal:** Navigate from the entrance (Location 1) through the passages, avoiding dead‑ends, and eventually reach the exit. When you get out, you should bring back the message that’s waiting there.
- **How it works:**  
  1. Start at the entrance page: **https://maze.edwarddonner.com/1**.  
  2. Each page tells you where you are and lists the available exits.  
  3. You choose an exit, follow it to the next page, and keep track of the rooms you’ve visited and the exits you’ve discovered.  
  4. When you hit a dead end, backtrack and try another path.  
  5. Keep a running “mini‑map” of t

In [16]:
messages

[{'role': 'user',
  'content': 'Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now without going into too much detal '},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call-365ec444-0afe-4dfa-8a27-8dfb772ff4fd', function=Function(arguments='{"url":"https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)], reasoning="The user wants me to read the web page at https://maze.edwarddonner.com and describe what it says without too much detail. I'll use the read_web_page tool.", reasoning_details=[{'type': 'reasoning.text', 'text': "The user wants me to read the web page at https://maze.edwarddonner.com and describe what it says without too much detail. I'll use the read_web_page tool.", 'format': 'unknown', 'index': 0}]),
 {'role': 'tool',
  'tool_call_id': 'call-365ec444-0afe-4dfa-8a27-8dfb772f

In [ ]:
#agent loop from first principles
from IPython.display import Markdown, display

def run_agent(task, max_steps=25):
  """Call the model in a loop running the toop , whever it asks, untill it gives a final answer"""
  messages=[
      {"role":"system", "content": SYSTEM_PROMPT},
      {"role":"user", "content":task}
  ]

  for step in range(1, max_steps+1):
    response=openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    message=response.choices[0].message
    print(f"Step {step} asnwered by {response.model}")
    text=(message.content or "").strip()
    if text:
      display(Markdown(text))
    if message.tool_calls:
      messages.append(message)
      for tool_call in message.tool_calls:
        messages.append(handle_tool_call(tool_call))
    elif text:
      return
    else:
      print("The reply was empty so we ask again")
  print("The agent ran out of steps")

SYSTEM_PROMPT = """You are an agent exploring a web maze at https://maze.edwarddonner.com

You have one tool, read_web_page, which fetches a maze page and returns its text.
You can only read URLs that start with https://maze.edwarddonner.com

How to work:
- Start by reading https://maze.edwarddonner.com
- Each page's text contains links to other maze pages. Follow them one at a time to work your way through the maze.
- Only ever request URLs that you have actually seen in a page's text (or the start URL above). Never invent or guess a URL.
- Keep track of the pages you have already visited. Never read the same page twice - if a page offers several links, pick one you have not visited yet.
- If you reach a dead end, go back to the most recent page that still had an unvisited link and try that one instead.
- Work step by step: one tool call, look at the result, then decide the next step.

When you reach the final page of the maze (it will make clear that it is the end), stop calling tools and give a final answer that:
- Quotes what the final page says
- Lists the path you took through the maze, in order
- Briefly describes what each page along the way said"""


In [ ]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says")

Step 1 asnwered by nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
Reading the web page: https://maze.edwarddonner.com
Step 2 asnwered by nvidia/nemotron-3.5-super-vl:free
Reading the web page: https://maze.edwarddonner.com/1
Step 3 asnwered by nvidia/nemotron-3.5-lightning:free
Reading the web page: https://maze.edwarddonner.com/14
Step 4 asnwered by nvidia/nemotron-3-ultra-550b-a55b:free
Reading the web page: https://maze.edwarddonner.com/62
Step 5 asnwered by nvidia/nemotron-3.5-lightning:free
Reading the web page: https://maze.edwarddonner.com/39
Step 6 asnwered by nvidia/nemotron-3-ultra-550b-a55b:free
Reading the web page: https://maze.edwarddonner.com/71
Step 7 asnwered by xingchen-agi/xing4.0-29b-a4b:free
Reading the web page: https://maze.edwarddonner.com/62
Step 8 asnwered by nvidia/nemotron-3-ultra-550b-a55b:free
Reading the web page: https://maze.edwarddonner.com/48
Step 9 asnwered by recursive/flint-v1.0:free
Reading the web page: https://maze.edwarddonner.com/83
Step 1

I've reached the end of the maze! Here's the final answer:

**Final page message:**
> "You climb the last steps and walk out of the maze into the daylight. Your message: Welcome to the start of the course! Prepare for 3 intensive weeks ahead. You are out of the maze. Give the person you are working with the message above and a map of the maze based on your journey, showing the rooms you passed through, the dead ends you found and the route that led out."

**Path through the maze (in order):**
1. **Intro page** – https://maze.edwarddonner.com (explains the maze rules and points to Location 1)
2. **Location 1 (Entrance)** – https://maze.edwarddonner.com/1 – "You are at location 1. This is the entrance. Two stone corridors lead away into the dark." Exits: North to 14, East to 27.
3. **Location 14** – https://maze.edwarddonner.com/14 – "A long corridor with damp walls. Someone has scratched a tally into the stone." Exits: North to 62, South back to 1.
4. **Location 62** – https://maze.edwarddonner.com/62 – "A junction with a low ceiling. A faint hum comes from somewhere to the east." Exits: North to 39, East to 48, South to 14.
5. **Location 39** – https://maze.edwarddonner.com/39 – "A chamber with an old wooden door that has been painted shut." Exits: East to 71, South to 62.
6. **Location 71 (Dead end)** – https://maze.edwarddonner.com/71 – "A cupboard of a room with a broom in it. Dead end." Only exit West back to 39.
7. **Back to Location 62** (revisited, then took East exit)
8. **Location 48** – https://maze.edwarddonner.com/48 – "A narrow bridge over a dry channel. The hum is louder here." Exits: East to 83, West to 62.
9. **Location 83** – https://maze.edwarddonner.com/83 – "A round chamber with three ways out. Cool air is moving through it." Exits: North to 25, East to 56, West to 48.
10. **Location 25** – https://maze.edwarddonner.com/25 – "A stone staircase going up. There is daylight at the top." Exits: East to 35, South to 83.
11. **Location 35 (Exit)** – https://maze.edwarddonner.com/35 – Final page with the success message.

**Brief descriptions of each room:**
- **Start page**: Instructions for the AI to explore the maze step by step.
- **Location 1**: Maze entrance with two corridors (North and East).
- **Location 14**: Damp corridor with a tally mark.
- **Location 62**: Junction with a humming sound; three exits.
- **Location 39**: Chamber with a painted-shut door; leads to a dead end.
- **Location 71**: Tiny room with a broom – dead end.
- **Location 48**: Bridge over a dry channel; humming louder.
- **Location 83**: Round chamber with three exits and cool air.
- **Location 25**: Staircase upward with daylight visible.
- **Location 35**: Exit into daylight with the final message.